# Final_02_post.ipynb 步驟目錄

---
| Step | 內容 | 使用資料 | 輸出檔案 |
|---:|---|---|---|
| 1 | 讀取 post-analysis 所需基礎資料 | `output/taipei_road_grid_segments.geojson`, `output/Taipei_grid_full_risk.geojson`, `data/*_flooding.gpkg`, `data/debris1753_20260126_twd97/*.shp` | `road_grid_segments`, `grid`, `flood_maps`, `debris_potential` |
| 2 | 讀取雨量資料 | `data/rain_20240710.csv` | `rain_tp` |
| 3 | 擷取指定時間的四種雨量測站資料 | `rain_tp`, `target_time` | `rain_past1hr_station`, `rain_past24hr_station`, `rain_past2days_station`, `rain_past3days_station` |
| 4 | 使用 Kriging 將四種雨量內插至 grid | 四種雨量測站資料、`grid` | `grid_rain` |
| 5 | 視覺化 Kriging 雨量與不確定度 | `grid_rain`, 四種雨量測站資料 | - |
| 6 | 輸出含雨量資訊的 grid 資料 | `grid_rain` | `output/Taipei_grid_with_rain_post.geojson`, `output/Taipei_grid_with_rain_post.csv` |
| 7 | 建立平均雨量模擬情境 | `grid_rain` | `grid_rain_mean` |
| 8 | 選擇災後分析使用的雨量模式 | `grid_rain`, `grid_rain_mean`, `RAIN_MODE` | `grid_rain_selected` |
| 9 | 定義道路車速、降雨、淹水、土石流與 tunnel 封閉規則 | 道路型態、雨量欄位、淹水圖層、土石流潛勢區 | 災害影響函式與道路旅行時間函式 |
| 10 | 建立 grid 災害影響欄位並計算 road-grid 災前 / 災後旅行時間 | `grid_rain_selected`, `flood_maps`, `debris_potential`, `road_grid_segments` | `grid_disaster`, `road_grid` |
| 11 | 輸出道路災前 / 災後旅行時間資料 | `road_grid` | `output/road_grid_travel_time_pre_post.geojson`, `output/road_grid_travel_time_pre_post.csv` |
---

## Input Data Loading
---

此步驟讀取新版 post-disaster road speed model 所需的所有基礎資料。道路分段資料來自第一階段輸出，並在此階段移除舊版風險判斷與災害影響欄位，避免和新版規則混用。

讀取資料包含：

| Variable name | File path | Description |
|---|---|---|
| `road_grid_segments` | `output/taipei_road_grid_segments.geojson` | 道路切分至 grid 後的 road-grid segment |
| `grid` | `output/Taipei_grid_full_risk.geojson` | 臺北市 grid geometry，用於雨量 Kriging 與套疊 |
| `flood_maps` | `data/78.8mm_flooding.gpkg`, `data/100mm_flooding.gpkg`, `data/130mm_flooding.gpkg` | 不同降雨情境下的淹水深度圖 |
| `debris_potential` | `data/debris1753_20260126_twd97/debris1753_20260126_twd97.shp` | 115年度 1753 條土石流潛勢溪流影響範圍 |


In [ ]:
# Cell 1: Load required data for Final_02_post

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

# ===== 0) Path settings =====
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_CRS = "EPSG:3826"

road_grid_path = OUTPUT_DIR / "taipei_road_grid_segments.geojson"
grid_path = OUTPUT_DIR / "Taipei_grid_full_risk.geojson"

flood_paths = {
    78.8: DATA_DIR / "78.8mm_flooding.gpkg",
    100.0: DATA_DIR / "100mm_flooding.gpkg",
    130.0: DATA_DIR / "130mm_flooding.gpkg",
}

debris_path = (
    DATA_DIR
    / "debris1753_20260126_twd97"
    / "debris1753_20260126_twd97.shp"
)

required_paths = [road_grid_path, grid_path, debris_path, *flood_paths.values()]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

# ===== 1) Read road-grid segments =====
road_grid_segments = gpd.read_file(road_grid_path).to_crs(TARGET_CRS)

# ===== 2) Read Taipei grid geometry =====
grid = gpd.read_file(grid_path).to_crs(TARGET_CRS)

# The new post-analysis only carries grid_id and geometry forward.
if "grid_id" not in grid.columns:
    raise KeyError("grid data must contain a 'grid_id' column")

grid = grid[["grid_id", "geometry"]].copy()

# ===== 3) Read flood-depth simulation maps =====
flood_maps = {
    scenario_mmhr: gpd.read_file(path).to_crs(TARGET_CRS)
    for scenario_mmhr, path in flood_paths.items()
}

# ===== 4) Read debris-flow potential influence area =====
debris_potential = gpd.read_file(
    debris_path,
    encoding="cp950"
).to_crs(TARGET_CRS)

# ===== 5) Quick check =====
print("road-grid segments:", len(road_grid_segments))
print("grid cells:", len(grid))

for scenario_mmhr, flood_gdf in flood_maps.items():
    print(f"flood map {scenario_mmhr} mm/hr:", len(flood_gdf))

print("debris-flow potential polygons:", len(debris_potential))

display(road_grid_segments.head())
display(grid.head())

## 讀取雨量資料
---
此步驟讀取 `data/rain_20240418.csv`，並將時間、雨量欄位與測站座標轉成可分析的格式。接著只保留 `CountyName` 為臺北市的測站資料，作為後續雨量內插與道路災害分析使用。

In [ ]:
rain_path = Path("data/rain_20240710.csv")

# 讀取雨量資料
rain = pd.read_csv(rain_path, encoding="utf-8-sig")

# 基本欄位清理
rain["CountyName"] = rain["CountyName"].astype(str).str.strip()
rain["DateTime"] = pd.to_datetime(rain["DateTime"], errors="coerce")

# 雨量與座標欄位轉數值
rain_cols = [
    "Past1hr",
    "Past10Min",
    "Past3hr",
    "Past6hr",
    "Past12hr",
    "Past24hr",
    "NOW",
    "Past2days",
    "Past3days",
]

num_cols = [
    "StationLatitude",
    "StationLongitude",
    "StationAltitude",
    *rain_cols,
]

for col in num_cols:
    rain[col] = pd.to_numeric(rain[col], errors="coerce")

# 只保留臺北市資料
rain_tp = rain[rain["CountyName"].isin(["臺北市", "台北市"])].copy()

print("臺北市雨量紀錄筆數:", len(rain_tp))
print("臺北市測站數:", rain_tp["StationId"].nunique())

display(rain_tp.head())

## 擷取指定時間雨量資料
---
此步驟設定分析時間為 `2024-07-10 14:00:00`，並擷取該時間點臺北市各測站的累積雨量欄位。雨量資料依災害影響類型分為兩組：`Past1hr` 用於 rain / flooding impact，`Past24hr`、`Past2days` 與 `Past3days` 則用於 landsliding / debris-flow impact。

| 雨量欄位 | 服務的災害影響 | 使用目的 |
|---|---|---|
| Past1hr | rain / flooding | 判斷短時間降雨強度，作為降雨與淹水影響道路速度的依據 |
| Past24hr | landsliding | 判斷近 24 小時累積雨量對坡地災害或土石流風險的影響 |
| Past2days | landsliding | 判斷連續兩日累積雨量對坡地災害或土石流風險的影響 |
| Past3days | landsliding | 判斷連續三日累積雨量對坡地災害或土石流風險的影響 |

In [ ]:
from pykrige.ok import OrdinaryKriging

# ===== 參數設定 =====
target_time = pd.Timestamp("2024-07-10 14:00:00")

# 確保格式正確
rain_tp = rain_tp.copy()
rain_tp["DateTime"] = pd.to_datetime(rain_tp["DateTime"])

rain_value_cols = [
    "Past1hr",
    "Past24hr",
    "Past2days",
    "Past3days",
    "StationLongitude",
    "StationLatitude",
]

for col in rain_value_cols:
    rain_tp[col] = pd.to_numeric(rain_tp[col], errors="coerce")

station_cols = [
    "StationId",
    "StationName",
    "CountyName",
    "TownName",
    "StationLatitude",
    "StationLongitude",
]

rain_at_time = rain_tp[
    rain_tp["DateTime"] == target_time
].copy()

# ===== 1) rain / flooding 判斷用：Past1hr =====
rain_past1hr_station = (
    rain_at_time
    .dropna(subset=["Past1hr", "StationLatitude", "StationLongitude"])
    [station_cols + ["Past1hr"]]
    .rename(columns={"Past1hr": "rain_past1hr_mm"})
    .reset_index(drop=True)
)

# ===== 2) landsliding / debris-flow 判斷用：Past24hr, Past2days, Past3days =====
rain_past24hr_station = (
    rain_at_time
    .dropna(subset=["Past24hr", "StationLatitude", "StationLongitude"])
    [station_cols + ["Past24hr"]]
    .rename(columns={"Past24hr": "rain_past24hr_mm"})
    .reset_index(drop=True)
)

rain_past2days_station = (
    rain_at_time
    .dropna(subset=["Past2days", "StationLatitude", "StationLongitude"])
    [station_cols + ["Past2days"]]
    .rename(columns={"Past2days": "rain_past2days_mm"})
    .reset_index(drop=True)
)

rain_past3days_station = (
    rain_at_time
    .dropna(subset=["Past3days", "StationLatitude", "StationLongitude"])
    [station_cols + ["Past3days"]]
    .rename(columns={"Past3days": "rain_past3days_mm"})
    .reset_index(drop=True)
)

# ===== 3) Quick check =====
rain_station_tables = {
    "Past1hr": (rain_past1hr_station, "rain_past1hr_mm"),
    "Past24hr": (rain_past24hr_station, "rain_past24hr_mm"),
    "Past2days": (rain_past2days_station, "rain_past2days_mm"),
    "Past3days": (rain_past3days_station, "rain_past3days_mm"),
}

for label, (df, value_col) in rain_station_tables.items():
    print(f"{label} 測站數:", len(df))
    print(
        f"{label} 雨量範圍:",
        df[value_col].min(),
        "~",
        df[value_col].max(),
    )

display(rain_past1hr_station.sort_values("rain_past1hr_mm", ascending=False).head(10))
display(rain_past24hr_station.sort_values("rain_past24hr_mm", ascending=False).head(10))
display(rain_past2days_station.sort_values("rain_past2days_mm", ascending=False).head(10))
display(rain_past3days_station.sort_values("rain_past3days_mm", ascending=False).head(10))

## Kriging 雨量內插至網格
---
此步驟使用 Ordinary Kriging 將臺北市測站雨量內插到每個 grid 中心點，讓每個網格都具有對應的雨量估計值與不確定度。內插後的雨量資料依災害模組使用：`Past1hr` 服務 rain / flooding impact，`Past24hr`、`Past2days` 與 `Past3days` 服務 landsliding / debris-flow impact。

| 雨量資料 | 輸入測站資料 | 輸出欄位 | 不確定度欄位 | 用途 |
|---|---|---|---|---|
| Past 1hr | `rain_past1hr_station` | `rain_past1hr_mm` | `rain_past1hr_std` | rain / flooding impact |
| Past 24hr | `rain_past24hr_station` | `rain_past24hr_mm` | `rain_past24hr_std` | landsliding / debris-flow impact |
| Past 2days | `rain_past2days_station` | `rain_past2days_mm` | `rain_past2days_std` | landsliding / debris-flow impact |
| Past 3days | `rain_past3days_station` | `rain_past3days_mm` | `rain_past3days_std` | landsliding / debris-flow impact |

In [ ]:
def kriging_to_grid(
    station_df,
    value_col,
    output_prefix,
    target_grid,
    variogram_model="spherical",
    nugget=0,
    sill=None,
    range_m=15000,
    n_closest_points=16,
):
    station_gdf = gpd.GeoDataFrame(
        station_df,
        geometry=gpd.points_from_xy(
            station_df["StationLongitude"],
            station_df["StationLatitude"],
        ),
        crs="EPSG:4326",
    ).to_crs(TARGET_CRS)

    x = station_gdf.geometry.x.to_numpy()
    y = station_gdf.geometry.y.to_numpy()
    z = station_gdf[value_col].to_numpy()

    if len(station_gdf) < 3:
        raise ValueError(f"{output_prefix}: 測站數少於 3，無法 kriging")

    if sill is None:
        sill = np.var(z)

    grid_centroid = target_grid.geometry.centroid
    grid_x = grid_centroid.x.to_numpy()
    grid_y = grid_centroid.y.to_numpy()

    if np.nanmax(z) == np.nanmin(z):
        pred = np.full(len(target_grid), z[0])
        var = np.zeros(len(target_grid))
    else:
        OK = OrdinaryKriging(
            x,
            y,
            z,
            variogram_model=variogram_model,
            variogram_parameters={
                "nugget": nugget,
                "sill": sill,
                "range": range_m,
            },
            coordinates_type="euclidean",
            verbose=False,
            enable_plotting=False,
        )

        pred, var = OK.execute(
            "points",
            grid_x,
            grid_y,
            n_closest_points=n_closest_points,
            backend="loop",
        )

        pred = np.asarray(pred, dtype=float)
        var = np.asarray(var, dtype=float)

    pred = np.clip(pred, 0, None)
    var = np.clip(var, 0, None)

    return pred, var


grid_rain = grid.copy()

rain_kriging_inputs = [
    {
        "label": "Past 1hr",
        "station_df": rain_past1hr_station,
        "value_col": "rain_past1hr_mm",
        "prefix": "rain_past1hr",
        "range_m": 10000,
        "n_closest_points": 12,
    },
    {
        "label": "Past 24hr",
        "station_df": rain_past24hr_station,
        "value_col": "rain_past24hr_mm",
        "prefix": "rain_past24hr",
        "range_m": 15000,
        "n_closest_points": 16,
    },
    {
        "label": "Past 2days",
        "station_df": rain_past2days_station,
        "value_col": "rain_past2days_mm",
        "prefix": "rain_past2days",
        "range_m": 15000,
        "n_closest_points": 16,
    },
    {
        "label": "Past 3days",
        "station_df": rain_past3days_station,
        "value_col": "rain_past3days_mm",
        "prefix": "rain_past3days",
        "range_m": 15000,
        "n_closest_points": 16,
    },
]

for item in rain_kriging_inputs:
    pred, var = kriging_to_grid(
        item["station_df"],
        item["value_col"],
        item["prefix"],
        target_grid=grid,
        variogram_model="spherical",
        nugget=0,
        sill=None,
        range_m=item["range_m"],
        n_closest_points=item["n_closest_points"],
    )

    grid_rain[f"{item['prefix']}_mm"] = pred
    grid_rain[f"{item['prefix']}_var"] = var
    grid_rain[f"{item['prefix']}_std"] = np.sqrt(np.clip(var, 0, None))

    mm_col = f"{item['prefix']}_mm"
    print(
        f"{item['label']} kriging 完成:",
        f"{grid_rain[mm_col].min():.2f}",
        "~",
        f"{grid_rain[mm_col].max():.2f}",
        "mm",
    )

display(
    grid_rain[
        [
            "grid_id",
            "rain_past1hr_mm",
            "rain_past1hr_std",
            "rain_past24hr_mm",
            "rain_past24hr_std",
            "rain_past2days_mm",
            "rain_past2days_std",
            "rain_past3days_mm",
            "rain_past3days_std",
        ]
    ].head()
)

## 視覺化
---

In [ ]:
import matplotlib.pyplot as plt

def station_to_gdf(station_df):
    return gpd.GeoDataFrame(
        station_df,
        geometry=gpd.points_from_xy(
            station_df["StationLongitude"],
            station_df["StationLatitude"],
        ),
        crs="EPSG:4326",
    ).to_crs(TARGET_CRS)


rain_past1hr_gdf = station_to_gdf(rain_past1hr_station)
rain_past24hr_gdf = station_to_gdf(rain_past24hr_station)
rain_past2days_gdf = station_to_gdf(rain_past2days_station)
rain_past3days_gdf = station_to_gdf(rain_past3days_station)

fig, axes = plt.subplots(4, 2, figsize=(14, 28), constrained_layout=True)

rain_layers = [
    {
        "rain_col": "rain_past1hr_mm",
        "std_col": "rain_past1hr_std",
        "station_gdf": rain_past1hr_gdf,
        "title": f"Past 1hr rainfall at {target_time:%Y-%m-%d %H:%M} (mm)",
        "std_title": "Past 1hr kriging uncertainty, std (mm)",
        "cmap": "Blues",
    },
    {
        "rain_col": "rain_past24hr_mm",
        "std_col": "rain_past24hr_std",
        "station_gdf": rain_past24hr_gdf,
        "title": f"Past 24hr rainfall at {target_time:%Y-%m-%d %H:%M} (mm)",
        "std_title": "Past 24hr kriging uncertainty, std (mm)",
        "cmap": "PuBuGn",
    },
    {
        "rain_col": "rain_past2days_mm",
        "std_col": "rain_past2days_std",
        "station_gdf": rain_past2days_gdf,
        "title": f"Past 2days rainfall at {target_time:%Y-%m-%d %H:%M} (mm)",
        "std_title": "Past 2days kriging uncertainty, std (mm)",
        "cmap": "YlGnBu",
    },
    {
        "rain_col": "rain_past3days_mm",
        "std_col": "rain_past3days_std",
        "station_gdf": rain_past3days_gdf,
        "title": f"Past 3days rainfall at {target_time:%Y-%m-%d %H:%M} (mm)",
        "std_title": "Past 3days kriging uncertainty, std (mm)",
        "cmap": "GnBu",
    },
]

for row, layer in enumerate(rain_layers):
    # ===== 雨量分布 =====
    grid_rain.plot(
        column=layer["rain_col"],
        ax=axes[row, 0],
        cmap=layer["cmap"],
        legend=True,
        linewidth=0,
    )

    layer["station_gdf"].plot(
        ax=axes[row, 0],
        column=layer["rain_col"],
        cmap=layer["cmap"],
        edgecolor="black",
        markersize=20,
    )

    axes[row, 0].set_title(layer["title"])
    axes[row, 0].set_axis_off()

    # ===== 不確定度分布 =====
    grid_rain.plot(
        column=layer["std_col"],
        ax=axes[row, 1],
        cmap="magma",
        legend=True,
        linewidth=0,
    )

    layer["station_gdf"].plot(
        ax=axes[row, 1],
        color="white",
        edgecolor="black",
        markersize=20,
    )

    axes[row, 1].set_title(layer["std_title"])
    axes[row, 1].set_axis_off()

plt.show()

## 輸出含雨量資訊的 Grid 資料
---
此步驟將 Kriging 產生的四種雨量結果寫回臺北市 grid。每個 `grid_id` 只保留一筆資料，但同時包含 `Past1hr`、`Past24hr`、`Past2days` 與 `Past3days` 四種雨量估計值及其不確定度。

其中 `Past1hr` 服務 rain / flooding impact，`Past24hr`、`Past2days` 與 `Past3days` 服務 landsliding / debris-flow impact。

| 欄位 | 說明 |
|---|---|
| `rain_past1hr_mm` | 該網格的 Past 1-hour 雨量估計值 |
| `rain_past1hr_std` | Past 1-hour 雨量估計的不確定度 |
| `rain_past24hr_mm` | 該網格的 Past 24-hour 雨量估計值 |
| `rain_past24hr_std` | Past 24-hour 雨量估計的不確定度 |
| `rain_past2days_mm` | 該網格的 Past 2-day 雨量估計值 |
| `rain_past2days_std` | Past 2-day 雨量估計的不確定度 |
| `rain_past3days_mm` | 該網格的 Past 3-day 雨量估計值 |
| `rain_past3days_std` | Past 3-day 雨量估計的不確定度 |
| `rain_target_time` | 本次分析使用的時間點 |

輸出檔案如下：

| 輸出檔案 | 說明 |
|---|---|
| `output/Taipei_grid_with_rain_post.geojson` | 含 geometry 的 grid 雨量空間資料 |
| `output/Taipei_grid_with_rain_post.csv` | 不含 geometry 的 grid 雨量屬性表 |

In [ ]:
# ===== 保留四種雨量資訊 =====
rain_output_cols = [
    "rain_past1hr_mm",
    "rain_past1hr_std",
    "rain_past24hr_mm",
    "rain_past24hr_std",
    "rain_past2days_mm",
    "rain_past2days_std",
    "rain_past3days_mm",
    "rain_past3days_std",
]

# 確認必要欄位都存在
missing_rain_output_cols = [
    col for col in rain_output_cols
    if col not in grid_rain.columns
]

if missing_rain_output_cols:
    raise KeyError(f"grid_rain 缺少雨量欄位: {missing_rain_output_cols}")

# 紀錄分析時間
grid_rain["rain_target_time"] = target_time

# 確認一個 grid_id 只有一筆
print("grid 數量:", len(grid_rain))
print("不重複 grid_id 數量:", grid_rain["grid_id"].nunique())

display(
    grid_rain[
        [
            "grid_id",
            *rain_output_cols,
            "rain_target_time",
        ]
    ].head()
)

# ===== 輸出 =====
grid_rain.to_file(
    "output/Taipei_grid_with_rain_post.geojson",
    driver="GeoJSON",
    encoding="utf-8",
)

grid_rain.drop(columns="geometry").to_csv(
    "output/Taipei_grid_with_rain_post.csv",
    index=False,
    encoding="utf-8-sig",
)

### 建立平均雨量模擬情境
---
此步驟建立一組自行設定的平均雨量情境 `grid_rain_mean`，讓所有 grid 使用相同的降雨條件進行災後道路旅行時間模擬。此情境不使用測站 kriging 結果，而是直接指定固定雨量值，因此可用來測試不同降雨條件下的道路車速折減、淹水影響與土石流潛勢影響。

| 雨量欄位 | 模擬值 | 用途 |
|---|---:|---|
| `rain_past1hr_mm` | 20 mm | 降雨造成的即時車速折減與淹水判斷 |
| `rain_past24hr_mm` | 100 mm | 坡地 / 土石流危害判斷 |
| `rain_past2days_mm` | 150 mm | 坡地 / 土石流危害判斷 |
| `rain_past3days_mm` | 200 mm | 坡地 / 土石流危害判斷 |

由於此情境是人工設定的模擬雨量，並非由 Kriging 內插產生，因此所有不確定度欄位 `*_std` 先設定為 `0`。

In [ ]:
# ============================================================
# Create mean rainfall simulation grid
# ============================================================

grid_rain_mean = grid_rain[["grid_id", "geometry"]].copy()

grid_rain_mean["rain_past1hr_mm"] = 20
grid_rain_mean["rain_past24hr_mm"] = 100
grid_rain_mean["rain_past2days_mm"] = 150
grid_rain_mean["rain_past3days_mm"] = 200

# 模擬雨量不是 kriging，所以 uncertainty 先設為 0
grid_rain_mean["rain_past1hr_std"] = 0
grid_rain_mean["rain_past24hr_std"] = 0
grid_rain_mean["rain_past2days_std"] = 0
grid_rain_mean["rain_past3days_std"] = 0

grid_rain_mean["rain_target_time"] = "simulation_mean"

display(
    grid_rain_mean[
        [
            "grid_id",
            "rain_past1hr_mm",
            "rain_past24hr_mm",
            "rain_past2days_mm",
            "rain_past3days_mm",
        ]
    ].head()
)

In [ ]:
# ============================================================
# Select rainfall mode for post-disaster analysis
# ============================================================

# 可選:
# "real" = 使用測站 kriging 的真實雨量 grid_rain
# "mean" = 使用你自訂的平均模擬雨量 grid_rain_mean
RAIN_MODE = "mean"

rain_mode_map = {
    "real": grid_rain,
    "mean": grid_rain_mean,
}

if RAIN_MODE not in rain_mode_map:
    raise ValueError(f"Unknown RAIN_MODE: {RAIN_MODE}")

grid_rain_selected = rain_mode_map[RAIN_MODE].copy()

print("Selected rainfall mode:", RAIN_MODE)
print("grid cells:", len(grid_rain_selected))

display(
    grid_rain_selected[
        [
            "grid_id",
            "rain_past1hr_mm",
            "rain_past24hr_mm",
            "rain_past2days_mm",
            "rain_past3days_mm",
        ]
    ].head()
)

## Road Speed Impact Model

Road speed after a disaster is still evaluated using three major impact components:

1. direct rainfall impact
2. flooding / standing-water impact
3. landslide / debris-flow impact

The model first calculates the rainfall-adjusted road speed, then applies flood-depth speed limits and landslide closure rules.

---

### Normal speed for all types of roads

| road_type | normal_speed_kph | FFS mph | Description |
|---|---:|---:|---|
| expressway | 80 | 49.7 | Expressways or high-capacity roads with faster through traffic |
| arterial | 60 | 37.3 | Major urban roads supporting cross-district movement |
| bridge | 50 | 31.1 | Bridge or viaduct segments |
| tunnel | 40 | 24.9 | Tunnels, underground roads, or underpasses |
| residential | 30 | 18.6 | Local streets and residential roads |
| service | 20 | 12.4 | Service roads, access roads, or low-priority local connections |

Bridge and tunnel road segments now have independent normal speeds. Tunnel road segments are still closed when `Past1hr > 78.8 mm/hr` because intense short-duration rainfall can quickly make tunnels or underpasses impassable.

---

### 1. Rainfall speed factor

`rainfall_speed_factor` represents the direct impact of recent rainfall intensity on road travel speed. In this project, Past 1-hour rainfall is used as rainfall intensity in `mm/hr`.

The rainfall-adjusted speed is calculated as:

$$
v_{\text{rain}} =
v_{\text{base}} \times f_{\text{rain}}
$$

where:

| Symbol | Meaning |
|---|---|
| `v_base` | normal road speed |
| `f_rain` | rainfall speed factor |
| `v_rain` | rainfall-adjusted speed |

Past 1-hour rainfall is divided into three bins:

| Rainfall bin | Past 1-hour rainfall intensity |
|---:|---:|
| 1 | `< 2.54 mm/hr` |
| 2 | `2.54-6.35 mm/hr` |
| 3 | `> 6.35 mm/hr` |

The rainfall speed factor differs by road type:

| road_type | FFS km/hr | FFS mph | Applied FFS range | < 2.54 mm/hr | 2.54-6.35 mm/hr | > 6.35 mm/hr |
|---|---:|---:|---|---:|---:|---:|
| expressway | 80 | 49.7 | 50 mph | 0.96 | 0.95 | 0.93 |
| arterial | 60 | 37.3 | 55 mph | 0.97 | 0.96 | 0.94 |
| bridge | 50 | 31.1 | 55 mph | 0.97 | 0.96 | 0.94 |
| tunnel | 40 | 24.9 | 55 mph | 0.97 | 0.96 | 0.94 |
| residential | 30 | 18.6 | 55 mph | 0.97 | 0.96 | 0.94 |
| service | 20 | 12.4 | 55 mph | 0.97 | 0.96 | 0.94 |

If `rain_past1hr_mm` is missing, the model assumes no direct rainfall speed penalty:

$$
f_{\text{rain}} = 1.00
$$

---

### 2. Flooding impacts on road speed

Flooding impact is determined by **standing-water depth** rather than rainfall intensity alone.

For rainfall scenarios below `78.8 mm/hr`, official flood-depth maps are not available. Therefore, only the direct rainfall speed adjustment is applied:

$$
v_{\text{final}} = v_{\text{rain}}
$$

For rainfall scenarios of `78.8`, `100`, and `130 mm/hr`, Taipei City Government flood simulation maps are used to obtain the standing-water depth of each road segment.

The flood-related speed limit is estimated using the depth-disruption function:

$$
v_{\text{flood}}(w) =
0.0009w^2 - 0.5529w + 86.9448
$$

where:

| Symbol | Meaning | Unit |
|---|---|---|
| `w` | standing-water depth | `mm` |
| `v_flood(w)` | maximum passable speed under standing water | `km/hr` |

Because the function is used as a speed cap, the final flood-related speed is constrained to a valid range:

$$
v_{\text{flood}}(w) =
\max \left(0, v_{\text{flood}}(w)\right)
$$

Roads with standing-water depth `>= 300 mm` are treated as closed.

| Standing-water depth | Maximum passable speed | Road condition |
|---:|---:|---|
| `0 mm` | No additional penalty | Normal condition |
| `50 mm` | `61.5 km/hr` | Slight impact |
| `100 mm` | `40.7 km/hr` | Moderate impact |
| `150 mm` | `24.3 km/hr` | Significant impact |
| `200 mm` | `12.4 km/hr` | Severe impact |
| `250 mm` | `5.0 km/hr` | Nearly impassable |
| `>= 300 mm` | `0 km/hr` | Road closure |

Additional tunnel closure rule:

| Condition | Applied road type | Effect |
|---|---|---|
| `Past1hr > 78.8 mm/hr` | `tunnel` roads, tunnels, or underpasses | Road closure |

When flood-depth data are available, the road speed after rainfall and flooding is:

$$
v_{\text{rain+flood}} =
\min \left(
v_{\text{rain}},
v_{\text{flood}}(w)
\right)
$$

---

### 3. Landslide and debris-flow impacts on road speed

Landslide impact is evaluated using the official debris-flow potential stream influence area. The `115年度 1753條土石流潛勢溪流影響範圍` layer is overlaid with the Taipei grid to identify grids located in potential debris-flow risk areas.

The official method considers rainfall from the previous seven days because antecedent rainfall affects debris-flow probability. Each previous day is discounted by a reduction coefficient of `0.7`.

Since the available rainfall dataset only contains rainfall records for the previous three days, the accumulated effective rainfall is simplified as:

$$
R_{t,\text{approx}} =
R_0 + 0.7R_1 + 0.7^2R_2 + 0.7^3R_3
$$

where:

| Symbol | Meaning |
|---|---|
| `R0` | rainfall on the current day |
| `R1` | rainfall one day before |
| `R2` | rainfall two days before |
| `R3` | rainfall three days before |
| `R_t,approx` | approximated effective rainfall for debris-flow risk |

Landslide or debris-flow impact is only applied to grids located inside the debris-flow potential influence area. If a road segment is not located in a potential debris-flow grid, no landslide speed penalty is applied:

$$
f_{\text{landslide}} = 1.00
$$

If a road segment is located in a potential debris-flow grid and the approximated effective rainfall reaches the debris-flow warning condition, the segment is treated as closed or not recommended for routing:

$$
f_{\text{landslide}} = 0.00
$$

---

### Final road speed

The final road speed combines the three major impact components.

First, direct rainfall reduces the base road speed:

$$
v_{\text{rain}} =
v_{\text{base}} \times f_{\text{rain}}
$$

Second, if flood-depth data are available, the model applies the lower value between rainfall-adjusted speed and flood-depth speed limit:

$$
v_{\text{rain+flood}} =
\min \left(
v_{\text{rain}},
v_{\text{flood}}(w)
\right)
$$

If flood-depth data are not available:

$$
v_{\text{rain+flood}} =
v_{\text{rain}}
$$

Finally, landslide or debris-flow closure is applied:

$$
v_{\text{final}} =
v_{\text{rain+flood}} \times f_{\text{landslide}}
$$

If `f_landslide = 0.00`, the road segment is considered closed. Otherwise, the final speed is determined by direct rainfall reduction and, when available, flood-depth speed limitation.

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd

# ============================================================
# Parameters
# ============================================================

# Landslide / debris-flow rainfall threshold
LANDSLIDE_WARNING_RAIN_MM = 500

# Flood simulation scenarios
FLOOD_SCENARIOS = [78.8, 100.0, 130.0]
FLOOD_SCENARIO_METHOD = "floor"
FLOOD_DEPTH_METHOD = "mid"

# Tunnel rule:
# tunnel and underground are treated as the same road class: tunnel.
# If Past1hr rainfall is greater than 78.8 mm/hr, tunnel roads are closed.
TUNNEL_RAIN_CLOSURE_MMHR = 78.8

TUNNEL_ROAD_TYPES = {
    "tunnel",
    "underground",
    "underground_road",
    "underpass",
}

TUNNEL_OSM_VALUES = {
    "yes",
    "true",
    "1",
    "building_passage",
}

NORMAL_SPEED_KPH = {
    "expressway": 80,
    "arterial": 60,
    "bridge": 50,
    "tunnel": 40,
    "residential": 30,
    "service": 20,
}

RAIN_FACTORS = {
    "expressway": {
        "low": 0.96,
        "mid": 0.95,
        "high": 0.93,
    },
    "arterial": {
        "low": 0.97,
        "mid": 0.96,
        "high": 0.94,
    },
    "bridge": {
        "low": 0.97,
        "mid": 0.96,
        "high": 0.94,
    },
    "tunnel": {
        "low": 0.97,
        "mid": 0.96,
        "high": 0.94,
    },
    "residential": {
        "low": 0.97,
        "mid": 0.96,
        "high": 0.94,
    },
    "service": {
        "low": 0.97,
        "mid": 0.96,
        "high": 0.94,
    },
}


# ============================================================
# Rainfall speed reduction
# ============================================================

def get_rain_bin(rain_mm):
    if pd.isna(rain_mm):
        return "low"
    if rain_mm < 2.54:
        return "low"
    if rain_mm <= 6.35:
        return "mid"
    return "high"


def rainfall_speed_factor(road_type, rain_mm):
    road_type = str(road_type).strip().lower()

    if road_type not in RAIN_FACTORS:
        road_type = "residential"

    rain_bin = get_rain_bin(rain_mm)
    return RAIN_FACTORS[road_type][rain_bin]


# ============================================================
# Flood depth helper functions
# ============================================================

def parse_depth_to_mm(value, method=FLOOD_DEPTH_METHOD):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)]

    if not nums:
        return np.nan

    if len(nums) == 1:
        depth_m = nums[0]
    else:
        if method == "max":
            depth_m = max(nums)
        elif method == "min":
            depth_m = min(nums)
        else:
            depth_m = sum(nums) / len(nums)

    return depth_m * 1000


def flood_speed_cap_kph(depth_mm):
    if pd.isna(depth_mm) or depth_mm <= 0:
        return np.nan

    cap = 0.0009 * depth_mm**2 - 0.5529 * depth_mm + 86.9448
    return max(cap, 0)


def choose_flood_scenario(past1hr_mm):
    if pd.isna(past1hr_mm):
        return np.nan

    scenarios = sorted(FLOOD_SCENARIOS)

    if FLOOD_SCENARIO_METHOD == "nearest":
        return min(scenarios, key=lambda x: abs(x - past1hr_mm))

    if FLOOD_SCENARIO_METHOD == "floor":
        valid = [x for x in scenarios if x <= past1hr_mm]
        return max(valid) if valid else np.nan

    # Default: ceiling
    valid = [x for x in scenarios if x >= past1hr_mm]
    return min(valid) if valid else max(scenarios)


def find_flood_depth_column(flood_gdf):
    candidates = [
        "depth",
        "Depth",
        "DEPTH",
        "gridcode",
        "GRIDCODE",
        "Name",
        "name",
    ]

    for col in candidates:
        if col in flood_gdf.columns:
            return col

    raise KeyError(
        "Cannot find flood depth column. "
        f"Available columns: {list(flood_gdf.columns)}"
    )


def add_flood_depth_to_grid(grid_gdf, flood_maps):
    out = grid_gdf.copy()

    out["flood_scenario_mmhr"] = out["rain_past1hr_mm"].apply(
        choose_flood_scenario
    )

    out["standing_water_depth_mm"] = 0.0

    for scenario_mmhr, flood_gdf in flood_maps.items():
        scenario_mask = out["flood_scenario_mmhr"] == scenario_mmhr

        if scenario_mask.sum() == 0:
            continue

        flood_layer = flood_gdf.copy()
        depth_col = find_flood_depth_column(flood_layer)

        flood_layer["standing_water_depth_mm"] = flood_layer[depth_col].apply(
            parse_depth_to_mm
        )

        flood_layer = flood_layer[
            ["standing_water_depth_mm", "geometry"]
        ].dropna(subset=["standing_water_depth_mm"])

        if flood_layer.empty:
            continue

        joined = gpd.sjoin(
            out.loc[scenario_mask, ["grid_id", "geometry"]],
            flood_layer,
            how="left",
            predicate="intersects",
        )

        depth_by_grid = (
            joined.groupby("grid_id")["standing_water_depth_mm"]
            .max()
            .fillna(0)
        )

        out.loc[scenario_mask, "standing_water_depth_mm"] = (
            out.loc[scenario_mask, "grid_id"]
            .map(depth_by_grid)
            .fillna(0)
            .to_numpy()
        )

    out["flood_speed_cap_kph"] = out["standing_water_depth_mm"].apply(
        flood_speed_cap_kph
    )

    out["flood_closure_factor"] = np.where(
        out["standing_water_depth_mm"] >= 300,
        0.0,
        1.0,
    )

    out["is_closed_by_flood_grid"] = out["flood_closure_factor"] == 0

    return out


# ============================================================
# Debris-flow / landslide helper functions
# ============================================================

def add_debris_potential_to_grid(grid_gdf, debris_gdf):
    out = grid_gdf.copy()

    joined = gpd.sjoin(
        out[["grid_id", "geometry"]],
        debris_gdf[["geometry"]],
        how="left",
        predicate="intersects",
    )

    has_debris = joined.groupby("grid_id")["index_right"].apply(
        lambda s: s.notna().any()
    )

    out["has_debris_potential"] = (
        out["grid_id"].map(has_debris).fillna(False).astype(bool)
    )

    return out


def add_landslide_effective_rain_to_grid(grid_gdf):
    out = grid_gdf.copy()

    rain_cols = [
        "rain_past24hr_mm",
        "rain_past2days_mm",
        "rain_past3days_mm",
    ]

    missing_cols = [col for col in rain_cols if col not in out.columns]
    if missing_cols:
        raise KeyError(f"Missing landslide rainfall columns: {missing_cols}")

    out["landslide_effective_rain_mm"] = out[rain_cols].max(axis=1)

    return out


def landslide_factor(row):
    if not bool(row["has_debris_potential"]):
        return 1.0

    if pd.isna(row["landslide_effective_rain_mm"]):
        return 1.0

    if row["landslide_effective_rain_mm"] >= LANDSLIDE_WARNING_RAIN_MM:
        return 0.0

    return 1.0


# ============================================================
# Main function: add disaster impact fields to grid
# ============================================================

def add_disaster_factors_to_grid(
    grid_gdf,
    flood_maps,
    debris_gdf,
):
    out = grid_gdf.copy()

    required_cols = [
        "grid_id",
        "rain_past1hr_mm",
        "rain_past24hr_mm",
        "rain_past2days_mm",
        "rain_past3days_mm",
        "geometry",
    ]

    missing_cols = [col for col in required_cols if col not in out.columns]
    if missing_cols:
        raise KeyError(f"Missing required grid columns: {missing_cols}")

    out["rain_bin"] = out["rain_past1hr_mm"].apply(get_rain_bin)

    for road_type in NORMAL_SPEED_KPH:
        out[f"rain_factor_{road_type}"] = out["rain_past1hr_mm"].apply(
            lambda rain_mm, rt=road_type: rainfall_speed_factor(rt, rain_mm)
        )

    out = add_flood_depth_to_grid(out, flood_maps)
    out = add_debris_potential_to_grid(out, debris_gdf)
    out = add_landslide_effective_rain_to_grid(out)

    out["landslide_factor"] = out.apply(landslide_factor, axis=1)
    out["is_closed_by_landslide_grid"] = out["landslide_factor"] == 0

    out["is_closed_grid"] = (
        out["is_closed_by_flood_grid"]
        | out["is_closed_by_landslide_grid"]
    )

    return out

# ============================================================
# Road-grid travel time helper functions
# ============================================================

def normalize_road_type(value):
    if isinstance(value, list):
        value = value[0] if value else None
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if value in TUNNEL_ROAD_TYPES or value in TUNNEL_OSM_VALUES:
        return "tunnel"
    if value in NORMAL_SPEED_KPH:
        return value
    if value in ["motorway", "motorway_link", "trunk", "trunk_link"]:
        return "expressway"
    if value in [
        "primary",
        "primary_link",
        "secondary",
        "secondary_link",
        "tertiary",
        "tertiary_link",
    ]:
        return "arterial"
    if value == "service":
        return "service"
    if value in ["residential", "living_street", "unclassified", "road"]:
        return "residential"

    return None


def is_tunnel_value(value):
    values = value if isinstance(value, list) else [value]

    for item in values:
        if pd.isna(item):
            continue
        text = str(item).strip().lower()
        if text in TUNNEL_ROAD_TYPES or text in TUNNEL_OSM_VALUES:
            return True

    return False


def is_tunnel_road(row):
    for col in ["road_type", "road_type_id", "highway", "highway_type", "tunnel"]:
        if col in row.index and is_tunnel_value(row[col]):
            return True

    return False


def infer_segment_road_type(row):
    for col in ["road_type", "road_type_id", "highway", "highway_type"]:
        if col in row.index:
            road_type = normalize_road_type(row[col])
            if road_type is not None:
                return road_type

    return "residential"


def travel_time_seconds(length_m, speed_kph):
    speed_mps = speed_kph * 1000 / 3600
    return np.where(speed_mps > 0, length_m / speed_mps, np.nan)


def calculate_road_travel(road_grid_segments, grid_disaster):
    road_travel = road_grid_segments.copy()

    if "segment_length_m" in road_travel.columns:
        length_col = "segment_length_m"
    elif "length" in road_travel.columns:
        length_col = "length"
    else:
        raise KeyError("road_grid_segments must contain 'segment_length_m' or 'length'")

    road_travel["segment_length_m"] = pd.to_numeric(
        road_travel[length_col],
        errors="coerce",
    )

    grid_factor_cols = [
        "grid_id",
        "rain_past1hr_mm",
        "rain_bin",
        "rain_factor_expressway",
        "rain_factor_arterial",
        "rain_factor_bridge",
        "rain_factor_tunnel",
        "rain_factor_residential",
        "rain_factor_service",
        "flood_scenario_mmhr",
        "standing_water_depth_mm",
        "flood_speed_cap_kph",
        "flood_closure_factor",
        "is_closed_by_flood_grid",
        "rain_past24hr_mm",
        "rain_past2days_mm",
        "rain_past3days_mm",
        "landslide_effective_rain_mm",
        "has_debris_potential",
        "landslide_factor",
        "is_closed_by_landslide_grid",
        "is_closed_grid",
    ]

    missing_grid_cols = [
        col for col in grid_factor_cols if col not in grid_disaster.columns
    ]
    if missing_grid_cols:
        raise KeyError(f"grid_disaster is missing columns: {missing_grid_cols}")

    if grid_disaster["grid_id"].duplicated().any():
        raise ValueError("grid_disaster has duplicated grid_id")

    road_travel = road_travel.merge(
        grid_disaster[grid_factor_cols],
        on="grid_id",
        how="left",
    )

    road_travel["road_type_model"] = road_travel.apply(
        infer_segment_road_type,
        axis=1,
    )
    road_travel["is_tunnel_road"] = road_travel.apply(is_tunnel_road, axis=1)
    road_travel["pre_speed_kph"] = road_travel["road_type_model"].map(
        NORMAL_SPEED_KPH
    )

    def select_rain_factor(row):
        factor_col = f"rain_factor_{row['road_type_model']}"
        if factor_col in row.index and pd.notna(row[factor_col]):
            return row[factor_col]
        return 1.0

    road_travel["rain_factor"] = road_travel.apply(select_rain_factor, axis=1)
    road_travel["speed_after_rain_kph"] = (
        road_travel["pre_speed_kph"] * road_travel["rain_factor"]
    )

    road_travel["flood_speed_cap_kph"] = pd.to_numeric(
        road_travel["flood_speed_cap_kph"],
        errors="coerce",
    )
    road_travel["speed_after_rain_flood_kph"] = np.where(
        road_travel["flood_speed_cap_kph"].notna(),
        np.minimum(
            road_travel["speed_after_rain_kph"],
            road_travel["flood_speed_cap_kph"],
        ),
        road_travel["speed_after_rain_kph"],
    )

    road_travel["landslide_factor"] = pd.to_numeric(
        road_travel["landslide_factor"],
        errors="coerce",
    ).fillna(1.0)
    road_travel["post_speed_kph"] = (
        road_travel["speed_after_rain_flood_kph"]
        * road_travel["landslide_factor"]
    )

    road_travel["rain_past1hr_mm"] = pd.to_numeric(
        road_travel["rain_past1hr_mm"],
        errors="coerce",
    )
    road_travel["is_closed_by_tunnel_rain"] = (
        road_travel["is_tunnel_road"]
        & (road_travel["rain_past1hr_mm"] > TUNNEL_RAIN_CLOSURE_MMHR)
    )

    road_travel["is_closed_by_flood"] = (
        (road_travel["standing_water_depth_mm"].fillna(0) >= 300)
        | road_travel["is_closed_by_tunnel_rain"]
    )
    road_travel["is_closed_by_landslide"] = road_travel["landslide_factor"] == 0
    road_travel["is_closed_post"] = (
        road_travel["is_closed_by_flood"]
        | road_travel["is_closed_by_landslide"]
        | (road_travel["post_speed_kph"] <= 0)
    )
    road_travel.loc[road_travel["is_closed_post"], "post_speed_kph"] = 0.0

    road_travel["pre_travel_time_sec"] = travel_time_seconds(
        road_travel["segment_length_m"],
        road_travel["pre_speed_kph"],
    )
    road_travel["post_travel_time_sec"] = travel_time_seconds(
        road_travel["segment_length_m"],
        road_travel["post_speed_kph"],
    )
    road_travel["pre_travel_time_min"] = road_travel["pre_travel_time_sec"] / 60
    road_travel["post_travel_time_min"] = road_travel["post_travel_time_sec"] / 60

    road_travel["travel_time_increase_sec"] = (
        road_travel["post_travel_time_sec"]
        - road_travel["pre_travel_time_sec"]
    )
    road_travel["travel_time_increase_min"] = (
        road_travel["travel_time_increase_sec"] / 60
    )
    road_travel["travel_time_ratio"] = (
        road_travel["post_travel_time_sec"]
        / road_travel["pre_travel_time_sec"]
    )
    road_travel["travel_time_increase_rate_pct"] = (
        (road_travel["travel_time_ratio"] - 1) * 100
    )

    road_travel.loc[
        road_travel["is_closed_post"],
        [
            "post_travel_time_sec",
            "post_travel_time_min",
            "travel_time_increase_sec",
            "travel_time_increase_min",
            "travel_time_ratio",
            "travel_time_increase_rate_pct",
        ],
    ] = np.nan

    return road_travel



### 建立災前災後道路旅行時間資料
---
此步驟會依照目前選定的雨量模式 `RAIN_MODE`，先將雨量資料套用到每個 grid，產生淹水、土石流與降雨車速折減因子。接著再把 grid 層級的災害影響結果合併至 `road_grid_segments`，計算每個 road segment 的災前與災後車速、旅行時間、旅行時間增加率，以及是否封閉。

輸出結果會作為後續路網地圖、可達性分析與最佳路徑比較使用。

| 輸出資料 | 說明 |
|---|---|
| `grid_disaster` | 每個 grid 的雨量、淹水深度、土石流影響與封閉狀態 |
| `road_grid` | 每個 road-grid segment 的災前 / 災後速度與旅行時間 |
| `output/road_grid_travel_time_pre_post.geojson` | 含 geometry 的道路災前 / 災後旅行時間資料 |
| `output/road_grid_travel_time_pre_post.csv` | 不含 geometry 的道路災前 / 災後旅行時間屬性表 |

In [ ]:
# ============================================================
# Build selected post-disaster road-grid travel output
# ============================================================

# ===== 1) Add disaster impact fields to the selected rainfall grid =====
grid_disaster = add_disaster_factors_to_grid(
    grid_gdf=grid_rain_selected,
    flood_maps=flood_maps,
    debris_gdf=debris_potential,
)

grid_summary_cols = [
    "grid_id",
    "rain_past1hr_mm",
    "rain_bin",
    "rain_factor_expressway",
    "rain_factor_arterial",
    "rain_factor_bridge",
    "rain_factor_tunnel",
    "rain_factor_residential",
    "rain_factor_service",
    "flood_scenario_mmhr",
    "standing_water_depth_mm",
    "flood_speed_cap_kph",
    "flood_closure_factor",
    "rain_past24hr_mm",
    "rain_past2days_mm",
    "rain_past3days_mm",
    "landslide_effective_rain_mm",
    "has_debris_potential",
    "landslide_factor",
    "is_closed_grid",
]

print("rain mode:", RAIN_MODE)
print("grid cells:", len(grid_disaster))
print("closed by flood grid:", int(grid_disaster["is_closed_by_flood_grid"].sum()))
print("closed by landslide grid:", int(grid_disaster["is_closed_by_landslide_grid"].sum()))
print("closed total grid:", int(grid_disaster["is_closed_grid"].sum()))

display(grid_disaster[grid_summary_cols].head(100))

# ===== 2) Apply selected disaster factors to road-grid segments =====
road_grid = calculate_road_travel(
    road_grid_segments=road_grid_segments,
    grid_disaster=grid_disaster,
)
road_grid["rain_mode"] = RAIN_MODE

road_summary_cols = [
    "road_grid_id",
    "grid_id",
    "road_type_model",
    "segment_length_m",
    "rain_past1hr_mm",
    "rain_factor",
    "pre_speed_kph",
    "post_speed_kph",
    "pre_travel_time_min",
    "post_travel_time_min",
    "travel_time_increase_rate_pct",
    "is_closed_post",
]

print("road-grid segments:", len(road_grid))
print("closed road segments:", int(road_grid["is_closed_post"].sum()))
print(
    "affected road segments:",
    int((road_grid["travel_time_increase_sec"].fillna(0) > 0).sum()),
)

display(road_grid[road_summary_cols].head(20))

# ===== 3) Save road-grid pre/post travel time =====
road_grid.to_file(
    OUTPUT_DIR / "road_grid_travel_time_pre_post.geojson",
    driver="GeoJSON",
    encoding="utf-8",
)

road_grid.drop(columns="geometry").to_csv(
    OUTPUT_DIR / "road_grid_travel_time_pre_post.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_DIR / "road_grid_travel_time_pre_post.geojson")
print("Saved:", OUTPUT_DIR / "road_grid_travel_time_pre_post.csv")
